# MainCrafts Technology – AI & ML Internship
## Task 1: Build & Evaluate a Linear Regression Model (House Price Predictor)

**Student:** Ankith M  
**Project:** California Housing Price Prediction using Linear Regression

### Objective
Build a complete machine-learning workflow: data loading, exploratory data analysis (EDA), preprocessing, train/test split, Linear Regression training, evaluation using MAE/RMSE/R², visualization, and model saving.

This notebook follows the task guide provided by MainCrafts Technology. The required deliverables are a Jupyter Notebook and a short PDF report.

## 1. Install / import libraries

If the packages are not installed, run this in a terminal:

```bash
pip install pandas numpy scikit-learn matplotlib seaborn jupyter joblib
```

The task specifically calls for Python, pandas, scikit-learn, EDA/visualization, train/test split, `LinearRegression`, and MAE/RMSE/R² evaluation.

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

## 2. Load the California Housing dataset

In [ ]:
# Load the built-in California Housing dataset
data = fetch_california_housing(as_frame=True)

df = data.frame.copy()
df = df.rename(columns={"MedHouseVal": "MedHouseVal"})

print("Dataset shape:", df.shape)
display(df.head())

### Dataset overview

The California Housing dataset contains 20,640 observations and 8 numeric predictive features. The target is `MedHouseVal`, the median house value expressed in units of $100,000.

Features:
- `MedInc` – median income
- `HouseAge` – median house age
- `AveRooms` – average rooms per household
- `AveBedrms` – average bedrooms per household
- `Population` – block-group population
- `AveOccup` – average household occupancy
- `Latitude` – latitude
- `Longitude` – longitude

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic structure and summary statistics
print("Data types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nSummary statistics:")
display(df.describe().T)

In [ ]:
# Check missing values and duplicates
print("Missing values by column:")
display(df.isnull().sum())

print("Total duplicate rows:", df.duplicated().sum())

In [ ]:
# Histograms for all variables
df.hist(figsize=(14, 10), bins=30)
plt.suptitle("California Housing Feature Distributions", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Target distribution
plt.figure(figsize=(8, 5))
sns.histplot(df["MedHouseVal"], bins=40, kde=True)
plt.title("Distribution of Median House Value")
plt.xlabel("MedHouseVal ($100,000s)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Median income vs house value
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="MedInc", y="MedHouseVal", alpha=0.25)
plt.title("Median Income vs Median House Value")
plt.xlabel("Median Income")
plt.ylabel("Median House Value ($100,000s)")
plt.tight_layout()
plt.show()

### EDA observations

1. The dataset is entirely numeric, so no categorical encoding is required.
2. Missing-value checks should show no missing values in the standard California Housing dataset.
3. `MedInc` generally has a strong positive relationship with `MedHouseVal`.
4. Geographic variables (`Latitude` and `Longitude`) also carry useful information.
5. The target contains a visible upper cap, so a simple linear model may not explain all nonlinear relationships.

## 4. Prepare features and target

In [ ]:
# Separate input features (X) and target (y)
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## 5. Train/test split

In [ ]:
# Use 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 6. Train the Linear Regression model

In [ ]:
# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

In [ ]:
# Display model coefficients
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", key=abs, ascending=False)

display(coefficients)

print("Intercept:", model.intercept_)

## 7. Make predictions and evaluate

In [ ]:
# Predict on unseen test data
y_pred = model.predict(X_test)

# Regression metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

### Metric interpretation

- **MAE (Mean Absolute Error):** average absolute prediction error. Lower is better.
- **RMSE (Root Mean Squared Error):** penalizes larger errors more strongly. Lower is better.
- **R² (R-squared):** proportion of target variance explained by the model. Higher is better; 1.0 is a perfect fit.

## 8. Actual vs Predicted plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.35)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.xlabel("Actual MedHouseVal")
plt.ylabel("Predicted MedHouseVal")
plt.title("Actual vs Predicted House Values")
plt.tight_layout()
plt.show()

## 9. Residual analysis

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted MedHouseVal")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.tight_layout()
plt.show()

print("Mean residual:", residuals.mean())

## 10. Save the trained model

In [ ]:
# Save model for optional future prediction/UI use
model_path = "california_housing_linear_regression.joblib"
joblib.dump(model, model_path)

print(f"Model saved as: {model_path}")

## 11. Final result summary

A standard run with `test_size=0.20` and `random_state=42` should produce results close to:

| Metric | Reference value |
|---|---:|
| MAE | 0.5332 |
| RMSE | 0.7456 |
| R² | 0.5758 |

These values are expressed in the dataset's target units, where 1.0 corresponds to $100,000 of median house value.

**Important:** Run all cells before submission and use the values printed by your own notebook in the final report.

## 12. Improvement ideas

The task only requires Linear Regression, but the model can be improved in future work by:

1. Testing Ridge/Lasso regression.
2. Trying nonlinear models such as Decision Tree, Random Forest, or Gradient Boosting.
3. Using cross-validation for more reliable evaluation.
4. Engineering additional features where appropriate.
5. Comparing models using RMSE, MAE, and R².
6. Building a small Streamlit UI for entering housing features and generating predictions.